[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/01_basic_classification/01_basic_classification.ipynb)

# 01. Basic Classification — scikit-learn 파이프라인 한 바퀴 (Iris 데이터셋)

> `00_python_essentials`에서 NumPy/Pandas 기본기를 익혔다면, 이 노트북에서는 그 도구들로
> **데이터 로드 → 탐색(EDA) → 전처리 → 모델 학습 → 평가 → 저장**이라는 scikit-learn 기반
> 머신러닝 파이프라인 전체 흐름을 한 번 훑어봅니다. 이론(Logistic/Softmax Regression의 수식 등)은
> 다음 노트북인 `03_classification`에서 다루므로, 여기서는 "실무에서 분류 모델을 어떻게 다루는지"
> 흐름 자체에 집중합니다.

## Colab에서 열기
1. 이 노트북 파일을 [colab.research.google.com](https://colab.research.google.com) → **파일 → 노트북 업로드**로 업로드하거나,
2. GitHub에 이 프로젝트를 push한 뒤 Colab의 **GitHub 탭**에서 저장소 URL로 불러오면 됩니다.

아래 첫 셀을 실행하면 Colab인지 로컬인지를 자동 감지해 필요한 패키지를 설치하고, 원한다면 Google Drive도 마운트합니다.


In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q scikit-learn pandas matplotlib seaborn joblib

    # 필요 시 주석 해제해서 Google Drive 마운트 (모델/데이터 저장용)
    # from google.colab import drive
    # drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. 데이터 로드

[Iris(붓꽃) 데이터셋](https://en.wikipedia.org/wiki/Iris_flower_data_set)은 꽃잎/꽃받침의 길이·너비 4개 특성(feature)으로
품종(setosa/versicolor/virginica) 3개 클래스를 맞히는, 분류 실습용으로 가장 널리 쓰이는 예제 데이터입니다.
`sklearn.datasets.load_iris`가 내장 제공하므로 별도 다운로드 없이 바로 `DataFrame`으로 변환해 사용합니다.


In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
df.head()

## 2. 탐색적 데이터 분석 (EDA)

모델을 학습시키기 전에 데이터의 분포와 클래스 간 차이를 먼저 눈으로 확인합니다.
`describe()`로 각 특성의 통계 요약(평균/표준편차/최소·최대값)을 보고,
`pairplot`으로 두 특성씩 짝지어 산점도를 그려 품종별로 얼마나 잘 구분되는지 살펴봅니다.


In [ ]:
df.describe()

In [ ]:
sns.pairplot(df, hue="species", vars=iris.feature_names)
plt.show()

## 3. 학습/테스트 분리 및 전처리

- **`train_test_split`**: 전체 데이터를 학습용(80%)과 테스트용(20%)으로 나눕니다. `stratify=y`를 주면
  품종별 비율을 학습/테스트 세트에서 동일하게 유지합니다.
- **`StandardScaler`**: 특성마다 값의 범위가 다르면 거리 기반/경사 하강법 기반 모델이 특정 특성에 과도하게
  민감해질 수 있습니다. 평균 0, 표준편차 1로 맞춰(표준화) 이런 편향을 줄입니다. `fit`은 학습 데이터에만
  적용하고, 테스트 데이터에는 같은 기준으로 `transform`만 적용해 데이터 누수(leakage)를 막습니다.


In [ ]:
X = df[iris.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train.shape, X_test.shape

## 4. 모델 학습

`LogisticRegression`으로 3개 클래스를 분류합니다. 이진 분류용으로 알려져 있지만 scikit-learn은
내부적으로 Softmax(다항 로지스틱)를 사용해 다중 클래스도 자연스럽게 처리합니다. 수식과 원리는
`03_classification`에서 직접 구현하며 다룹니다. 여기서는 `fit(X_train_scaled, y_train)` 한 줄로
경사 하강법 기반 학습이 끝난다는 점만 확인하고 넘어갑니다.


In [ ]:
model = LogisticRegression(max_iter=200, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)

## 5. 평가

학습된 모델을 테스트 세트(모델이 한 번도 보지 못한 데이터)에 적용해 성능을 확인합니다.
- **Accuracy**: 전체 중 맞춘 비율.
- **`classification_report`**: 클래스별 precision/recall/f1-score — 클래스 불균형이 있을 때 accuracy만으로는
  놓치기 쉬운 부분을 보완해줍니다.
- **Confusion Matrix**: 실제 클래스(행) vs 예측 클래스(열)를 표로 보여줘, 어떤 품종끼리 헷갈렸는지
  구체적으로 알 수 있습니다.


In [ ]:
y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=iris.target_names, yticklabels=iris.target_names, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## 6. 모델 저장

학습이 끝난 모델과 스케일러를 `joblib`으로 파일에 직렬화(serialize)해 저장합니다. 이렇게 저장해두면
다음에 다시 학습시키지 않고도 `joblib.load()`로 불러와 바로 예측에 사용할 수 있습니다(모델을 사용하는
쪽에서는 학습 때 쓴 것과 동일한 스케일러로 입력을 변환해야 하므로 스케일러도 함께 저장합니다).

Colab에서는 세션이 끝나면 `/content`가 사라지므로, 결과를 보존하려면 Google Drive에 마운트하고 경로를
바꿔주세요.


In [ ]:
import os
os.makedirs("../../../models", exist_ok=True)
joblib.dump(model, "../../../models/iris_logreg.joblib")
joblib.dump(scaler, "../../../models/iris_scaler.joblib")
print("Saved.")

**해설/정답**: [01_basic_classification_solutions.ipynb](01_basic_classification_solutions.ipynb)


## 정리 & 연습 문제
- 이 노트북에서 다룬 파이프라인(로드 → EDA → 전처리 → 학습 → 평가 → 저장)은 이후 노트북에서도
  반복되는 기본 골격입니다. 데이터셋과 모델만 바뀐다고 생각하면 됩니다.
- 다른 모델(`RandomForestClassifier`, `SVC` 등)로 바꿔서 같은 파이프라인으로 학습해보고 정확도를 비교해보세요.
- `GridSearchCV`로 하이퍼파라미터를 튜닝해보세요.
- 자신만의 CSV 데이터셋을 `data/` 폴더에 넣고 같은 흐름으로 분류를 시도해보세요.
- 다음 노트북([02_linear_regression.ipynb](../02_linear_regression/02_linear_regression.ipynb))에서는
  분류가 아닌 연속값을 예측하는 회귀 문제로 넘어가고, Gradient Descent를 직접 구현해 원리를 살펴봅니다.
